In [44]:
import math

def coef(n):
    c_n = (-1)**n/(2**n*math.factorial(n))*math.sqrt(math.factorial(2*n+1)/(4*math.pi))
    return 1/((2*n+1)*c_n**2)



coef(2)

1.3404128655316454

In [99]:
import os
import math
import pandas as pd
import numpy as np
print(os.getcwd())
work_path = '/home/yli/MR_CDFT/MRCDFT_2BCorr/MRCDFT'
A=20
Nuclei='Ne'
para_path = os.path.join(work_path,f'examples/{A}{Nuclei}/{A}{Nuclei}_para.dat')
b23_path = os.path.join(work_path,f'examples/{A}{Nuclei}/{A}{Nuclei}_b23.dat')
output_dir = os.path.join(work_path,f'examples/{A}{Nuclei}/output')

# A=150
# Nuclei='Nd'
# para_path = os.path.join(work_path,f'examples/{A}{Nuclei}_term/{A}{Nuclei}_para.dat')
# b23_path = os.path.join(work_path,f'examples/{A}{Nuclei}_term/{A}{Nuclei}_b23.dat')
# output_dir = os.path.join(work_path,f'examples/{A}{Nuclei}_term/output')

/home/yli/MR_CDFT/MRCDFT_2BCorr


In [109]:
# extract eMax, nbeta, nphi from parameter file
with open(para_path, "r", encoding="utf-8") as f:
    for line in f:
        line_nocomment = line.split("!")[0]

        if "n0f" in line_nocomment:
            eMax = int(line_nocomment.split("=")[1].split()[0])
        if "nphi" in line_nocomment:
            nphi = int(line_nocomment.split("=")[1])
        if "nbeta" in line_nocomment:
            nbeta = int(line_nocomment.split("=")[1])
        if "AMP" in line_nocomment:
            AMP = int(line_nocomment.split("=")[1])
        if "PNP" in line_nocomment:
            PNP = int(line_nocomment.split("=")[1])
        if "Kernels" in line_nocomment:
            Kernels = int(line_nocomment.split("=")[1])
    if(AMP == 0): nbeta = 1
    if(PNP == 0): nphi = 1

eMax = 6; nphi = 7; nbeta = 12; AMP = 1; PNP = 1; Kernels = 2

print('eMax=',eMax,' nphi=',nphi,' nbeta=',nbeta,"AMP=",AMP,"PNP=",PNP," Kernels=",Kernels)




eMax= 6  nphi= 7  nbeta= 12 AMP= 1 PNP= 1  Kernels= 2


In [110]:
oneB_ME_path = os.path.join(output_dir,f'mScheme_1B_A{A}_eMax{eMax:02d}.me')
oneB_ME = pd.read_csv(oneB_ME_path,sep=r'\s+')
oneB_ME = oneB_ME.drop(columns=["n1", "n2","l1", "l2","2j1", "2j2","2j_m1", "2j_m2"])
print(oneB_ME_path)
oneB_ME

/home/yli/MR_CDFT/MRCDFT_2BCorr/MRCDFT/examples/20Ne/output/mScheme_1B_A20_eMax06.me


,ifg,m1,m2,r^2,r^4,r^2Y20,r^4Y20,r^4Y40,f2,Eps1B
0,1,1,1,4.11810,28.26461,-0.00000,-0.00000,-0.0,-0.00000,11.24613
1,1,1,2,0.00000,0.00000,-0.00000,-0.00000,-0.0,-0.00000,0.00000
2,1,1,3,0.00000,0.00000,-0.00000,-0.00000,-0.0,-0.00000,0.00000
3,1,1,4,0.00000,0.00000,-0.00000,-0.00000,-0.0,-0.00000,0.00000
4,1,1,5,0.00000,0.00000,-0.00000,-0.00000,-0.0,-0.00000,0.00000
...,...,...,...,...,...,...,...,...,...,...
85819,2,240,236,0.00000,0.00000,-0.00000,-0.00000,0.0,-0.00000,0.00000
85820,2,240,237,0.00000,0.00000,-0.00000,-0.00000,-0.0,-0.00000,0.00000
85821,2,240,238,0.00000,0.00000,-0.00000,-0.00000,-0.0,-4.16342,0.00000
85822,2,240,239,0.00000,0.00000,-0.00000,-0.00000,-0.0,-0.00000,0.00000


In [111]:
# Prepare result dataframe, add TD density filename and Kernel filenames

DensList = pd.DataFrame(columns=['beta2(qf)','beta3(qf)', 'beta2(qi)','beta3(qi)', 'Ddens_filename', 'Kernel_filename','Eccen_filename'])

b23_data = pd.read_csv(b23_path,
                    sep=r'\s+',   # 按空格分隔
                    skiprows=0)
for q1 in range(len(b23_data)):
    for q2 in range(len(b23_data)):
        if(Kernels == 2 and q1 != q2): continue
        beta2q1 = b23_data.iloc[q1]['beta2']
        beta3q1 = b23_data.iloc[q1]['beta3']
        beta2q2 = b23_data.iloc[q2]['beta2']
        beta3q2 = b23_data.iloc[q2]['beta3']

        TDdens_filename = f"Proj_{A}{Nuclei}_D1B.{AMP}D_eMax{eMax:02d}.{nphi:02d}.{nbeta:02d}{int(beta2q1*100):+04d}{int(beta3q1*100):+04d}_{int(beta2q2*100):+04d}{int(beta3q2*100):+04d}.me"
        if not os.path.exists(os.path.join(output_dir,TDdens_filename)):
            print(TDdens_filename," not exist !")
        
        if(q1 <= q2):
            Kernel_filename = f"Proj_{A}{Nuclei}_kern.{AMP}D_eMax{eMax:02d}.{nphi:02d}.{nbeta:02d}{int(beta2q1*100):+04d}{int(beta3q1*100):+04d}_{int(beta2q2*100):+04d}{int(beta3q2*100):+04d}.elem"
            Eccen_filename = f"Proj_{A}{Nuclei}_Eccen.{AMP}D_eMax{eMax:02d}.{nphi:02d}.{nbeta:02d}{int(beta2q1*100):+04d}{int(beta3q1*100):+04d}_{int(beta2q2*100):+04d}{int(beta3q2*100):+04d}.elem"
        else:
            Kernel_filename = f"Proj_{A}{Nuclei}_kern.{AMP}D_eMax{eMax:02d}.{nphi:02d}.{nbeta:02d}{int(beta2q2*100):+04d}{int(beta3q2*100):+04d}_{int(beta2q1*100):+04d}{int(beta3q1*100):+04d}.elem"
            Eccen_filename = f"Proj_{A}{Nuclei}_Eccen.{AMP}D_eMax{eMax:02d}.{nphi:02d}.{nbeta:02d}{int(beta2q1*100):+04d}{int(beta3q1*100):+04d}_{int(beta2q2*100):+04d}{int(beta3q2*100):+04d}.elem"
        if not os.path.exists(os.path.join(output_dir,Kernel_filename)):
            print(Kernel_filename," not exist !")
        if not os.path.exists(os.path.join(output_dir,Eccen_filename)):
            print(Eccen_filename," not exist !")
        

        DensList.loc[len(DensList)] = [beta2q1, beta3q1, beta2q2, beta3q2, TDdens_filename, Kernel_filename,Eccen_filename]
DensList

,beta2(qf),beta3(qf),beta2(qi),beta3(qi),Ddens_filename,Kernel_filename,Eccen_filename
0,0.0,0.0,0.0,0.0,Proj_20Ne_D1B.1D_eMax06.07.12+000+000_+000+000.me,Proj_20Ne_kern.1D_eMax06.07.12+000+000_+000+00...,Proj_20Ne_Eccen.1D_eMax06.07.12+000+000_+000+0...
1,0.1,0.0,0.1,0.0,Proj_20Ne_D1B.1D_eMax06.07.12+010+000_+010+000.me,Proj_20Ne_kern.1D_eMax06.07.12+010+000_+010+00...,Proj_20Ne_Eccen.1D_eMax06.07.12+010+000_+010+0...
2,0.2,0.0,0.2,0.0,Proj_20Ne_D1B.1D_eMax06.07.12+020+000_+020+000.me,Proj_20Ne_kern.1D_eMax06.07.12+020+000_+020+00...,Proj_20Ne_Eccen.1D_eMax06.07.12+020+000_+020+0...
3,0.3,0.0,0.3,0.0,Proj_20Ne_D1B.1D_eMax06.07.12+030+000_+030+000.me,Proj_20Ne_kern.1D_eMax06.07.12+030+000_+030+00...,Proj_20Ne_Eccen.1D_eMax06.07.12+030+000_+030+0...
4,0.4,0.0,0.4,0.0,Proj_20Ne_D1B.1D_eMax06.07.12+040+000_+040+000.me,Proj_20Ne_kern.1D_eMax06.07.12+040+000_+040+00...,Proj_20Ne_Eccen.1D_eMax06.07.12+040+000_+040+0...
5,0.5,0.0,0.5,0.0,Proj_20Ne_D1B.1D_eMax06.07.12+050+000_+050+000.me,Proj_20Ne_kern.1D_eMax06.07.12+050+000_+050+00...,Proj_20Ne_Eccen.1D_eMax06.07.12+050+000_+050+0...
6,0.6,0.0,0.6,0.0,Proj_20Ne_D1B.1D_eMax06.07.12+060+000_+060+000.me,Proj_20Ne_kern.1D_eMax06.07.12+060+000_+060+00...,Proj_20Ne_Eccen.1D_eMax06.07.12+060+000_+060+0...
7,0.7,0.0,0.7,0.0,Proj_20Ne_D1B.1D_eMax06.07.12+070+000_+070+000.me,Proj_20Ne_kern.1D_eMax06.07.12+070+000_+070+00...,Proj_20Ne_Eccen.1D_eMax06.07.12+070+000_+070+0...
8,0.8,0.0,0.8,0.0,Proj_20Ne_D1B.1D_eMax06.07.12+080+000_+080+000.me,Proj_20Ne_kern.1D_eMax06.07.12+080+000_+080+00...,Proj_20Ne_Eccen.1D_eMax06.07.12+080+000_+080+0...
9,0.9,0.0,0.9,0.0,Proj_20Ne_D1B.1D_eMax06.07.12+090+000_+090+000.me,Proj_20Ne_kern.1D_eMax06.07.12+090+000_+090+00...,Proj_20Ne_Eccen.1D_eMax06.07.12+090+000_+090+0...


In [112]:
import math
Res_data = DensList.copy()
for q1q2 in range(len(Res_data)):
    # Ddens_path = os.path.join(output_dir,Res_data.iloc[q1q2]['Ddens_filename'])
    Kernel_path = os.path.join(output_dir,Res_data.iloc[q1q2]['Kernel_filename'])
    Eccen_path = os.path.join(output_dir,Res_data.iloc[q1q2]['Eccen_filename'])
    # read D1B density
    # Ddens = pd.read_csv(Ddens_path, sep=r'\s+')
    J = 0; Pi = '+'; K1 = 0; K2 = 0
    # df_tmp = Ddens[(Ddens["J"]==J) & (Ddens["Pi"]==Pi) & (Ddens["K1"]==K1) & (Ddens["K2"]==K2)]
    # merged = pd.merge(df_tmp, oneB_ME, left_on=["ifg","m1", "m2"], right_on=["ifg","m2", "m1"], how="left")
    # merged = merged.drop(columns=['m1_y','m2_y']).rename(columns={'m1_x':'m1','m2_x':'m2'})

    # read norm kernel
    with open(Kernel_path, 'r') as f:
        lines = f.readlines()
        line = lines[J*4+1]
        Norm = float(line[:15].strip())
        Energy = float(line[30:45].strip())
        line = lines[J*4+4]
        r2_proton = float(line[:15].strip())
        line = lines[J*4+6]
        r2_neutron = float(line[:15].strip())
        r2 = (r2_proton + r2_neutron)/Norm
        Res_data.loc[q1q2,f'r^2'] = r2

    # # calculate r^2
    # total_neutron = (merged["neutron"] * merged["r^2"]).sum()
    # total_proton = (merged["proton"] * merged["r^2"]).sum()
    # Res_data.loc[q1q2,f'r^2(total)'] = total_neutron/Norm + total_proton/Norm
    
    # # calculate r^4
    # total_neutron = (merged["neutron"] * merged["r^4"]).sum()
    # total_proton = (merged["proton"] * merged["r^4"]).sum()
    # Res_data.loc[q1q2,f'r4(total)'] = total_neutron/Norm + total_proton/Norm

    # # calculate r^2 Y20
    # total_neutron = (merged["neutron"] * merged["r^2Y20"]).sum()
    # total_proton = (merged["proton"] * merged["r^2Y20"]).sum()
    # Res_data.loc[q1q2,f'r2Y20(total)'] = total_neutron/Norm + total_proton/Norm

    # # calculate r^4 Y20
    # total_neutron = (merged["neutron"] * merged["r^4Y20"]).sum()
    # total_proton = (merged["proton"] * merged["r^4Y20"]).sum()
    # Res_data.loc[q1q2,f'r4Y20(total)'] = total_neutron/Norm + total_proton/Norm

    # # calculate r^4 Y40
    # total_neutron = (merged["neutron"] * merged["r^4Y40"]).sum()
    # total_proton = (merged["proton"] * merged["r^4Y40"]).sum()
    # Res_data.loc[q1q2,f'r4Y40(total)'] = total_neutron/Norm + total_proton/Norm
    
    # # calculate r^2_perp = 2/3 r^2 - 4/3*sqrt(pi/5)r^2 Y20
    # Res_data['r^2_perp'] = 2.0/3.0 * Res_data['r^2(total)'] - 4.0/3.0*math.sqrt(math.pi/5.0)*Res_data['r2Y20(total)'] 
    # # calculate r^4_perp = 8/15 r^4 - 32/21*sqrt(pi/5)r^4 Y20 + 16/105*sqrt(pi)*r^4 Y40
    # Res_data['r^4_perp'] = 8.0/15.0 * Res_data['r4(total)'] - 32.0/21.0*math.sqrt(math.pi/5.0)*Res_data['r4Y20(total)'] + 16.0/105.0*math.sqrt(math.pi)*Res_data['r4Y40(total)']
     
    # read Eccen kernel
    with open(Eccen_path, 'r') as f:
        lines = f.readlines()
        line = lines[1]
        oneBody = float(line[:15].strip())
        twoBody = float(line[15:30].strip())
        line = lines[2]
        twoBody_direct = float(line[:15].strip())
        twoBody_exchange = float(line[15:30].strip())
        twoBody_kappa = float(line[30:45].strip())
        coef_cn = coef(2)
        # print('Norm=',Norm)
        Res_data.loc[q1q2,f'1B'] =  oneBody*coef_cn
        Res_data.loc[q1q2,f'2B'] =  twoBody*coef_cn
        Res_data.loc[q1q2,f'2B_direct'] =  twoBody_direct*coef_cn
        Res_data.loc[q1q2,f'2B_exchange'] =  twoBody_exchange*coef_cn
        Res_data.loc[q1q2,f'2B_kappa'] =  twoBody_kappa*coef_cn

Res_data.drop(columns=['Ddens_filename', 'Kernel_filename','Eccen_filename'])
print(Nuclei,A)

Ne 20


In [113]:
print(Nuclei,A)
# droped_data = Res_data.drop(columns=['Ddens_filename', 'Kernel_filename','Eccen_filename','beta3(qf)', 'beta2(qf)','Eccen_filename','beta3(qi)','r4(total)','r2Y20(total)','r4Y20(total)','r4Y40(total)'])
droped_data = Res_data.drop(columns=['Ddens_filename', 'Kernel_filename','Eccen_filename','beta3(qf)', 'beta2(qf)','Eccen_filename','beta3(qi)'])
print(droped_data)
droped_data.to_csv(os.path.join(work_path,'scripts/Fig',f'{Nuclei}{A}_eMax{eMax}_P{AMP}.csv'), index=False)

Ne 20
    beta2(qi)         r^2           1B           2B     2B_direct  \
0         0.0  154.719791   934.695935   -94.280232 -2.737476e-11   
1         0.1  154.677278   934.310057   -76.841131  1.731659e+01   
2         0.2  154.677607   935.172935   -17.715493  8.836394e+01   
3         0.3  155.525692   949.351648   107.133362  2.556387e+02   
4         0.4  157.805630   986.408166   308.828229  5.329827e+02   
5         0.5  161.334447  1042.900107   553.055863  8.529941e+02   
6         0.6  166.774978  1127.679720   859.696458  1.210811e+03   
7         0.7  172.760728  1221.189495  1222.920625  1.626627e+03   
8         0.8  179.081895  1319.436289  1644.786983  2.103696e+03   
9         0.9  185.691285  1420.635369  2129.548100  2.645736e+03   
10        1.0  192.210627  1516.791495  2658.729829  3.227634e+03   

    2B_exchange   2B_kappa  
0   -175.203087  80.922860  
1   -175.797708  81.639996  
2   -183.762883  77.683440  
3   -209.367262  60.861878  
4   -253.565795  29.